# 9 — Toy model with metals: Lyα + Si III from GRF densities

The HI and Si III densities are drawn from the **same** 3D Gaussian random field, with Si III scaled down by a constant factor. The optical depths of Lyα and Si III 1206 along a 32×32 grid of skewers are wrapped to the Lyα frame. Si III absorption by the same gas therefore appears shifted by Δv = c ln(λ_Lyα/λ_SiIII) ≈ −2271 km/s, and its correlation with Lyα imprints:
- **P1D:** oscillations ∝ cos(kΔv) in the Lyα–Si III cross term, the McDonald et al. signature;
- **ξ1D:** a peak at |Δv|.

The notebook then:
1. compares the toy P1D with DESI DR1 at z = 2.2;
2. sweeps the field parameters;
3. fits the McDonald and Ma et al. contamination models (`models/models.py`);
4. shows the 3D statistics of the same skewers;
5. ends with light recovery checks.

In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"  # must be set before importing jax
import logging
logging.basicConfig(level=logging.WARNING, format="%(asctime)s - %(levelname)s - %(message)s")

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

import skewerjax as sjx
import skewerjax.utils.plotting_tools as pt
from types import SimpleNamespace
import matplotlib.colors as mcolors
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology
from skewerjax.fields.fields import ScalarField, VectorField, Fields
from skewerjax.fields.toy_model_samplers import PowerLawGRF
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux
import skewerjax.optical_depth.optical_depth as od
from skewerjax.optical_depth.overflux import DeltaFieldBuilder
from skewerjax.summary_stats.P1D_xi1D import OverfluxStatistics1D
from skewerjax.summary_stats.P3D_xi3D import OverfluxStatistics3D
from skewerjax.summary_stats.two_point import xi3d_from_power, xi_from_power
from skewerjax.constants.physical_constants import phys
from skewerjax.models import models

plt.style.use("tableau-colorblind10")
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc("font", **font)
plt.rcParams.update(rcnew)
%matplotlib inline
# %matplotlib widget   # interactive figures

## Data: DESI DR1 P1D at z = 2.2 (Chaves-Montero et al. 2026), digitised k [s/km] and kP/π

In [ ]:
Jonas2026_x = np.asarray([0.0012103424520726075, 0.001710426016737391, 0.0022405113342219723, 0.0026967336406882466, 0.0032317105483064905, 0.003716793236561375, 0.004218507331270713, 0.004720058372975595, 0.005196173145985431, 0.005711909799077935, 0.006196503328319453, 0.006692021408859485, 0.007234824860691589, 0.007707515520607861, 0.00821346899343304, 0.008739967144819602, 0.009207603161597756, 0.009733122994957585, 0.010237935096751576, 0.010697581516311415, 0.011248863724375202, 0.011746175387964242, 0.012217072464831506, 0.01268682817066758, 0.01319946681667543, 0.013692702155153086, 0.01418707886466193, 0.01473575222465443, 0.015217410799815749, 0.01573249524089043, 0.01624921021200967, 0.01672907520412198, 0.01725997578662884, 0.01769777310359165, 0.018203237417403463, 0.01873022472780339, 0.019218568476147384, 0.019730717963141867, 0.020171124128175965, 0.020730558986462522, 0.021214663356690675, 0.021757303755518324, 0.022217928493104895, 0.02264920368988949, 0.023259348032561685, 0.023726331837322017, 0.024201305239300665, 0.024734325510865445, 0.025247616368891116])
Jonas2026_y = np.asarray([0.008702200282951175, 0.010605036156919673, 0.015611253374371708, 0.01789981585681677, 0.020231052528309375, 0.01894901896933219, 0.019569830478807133, 0.02324977718276436, 0.026511191625119498, 0.029135409525467774, 0.02720121897976352, 0.026378752285570163, 0.02845338682745897, 0.032664334384986116, 0.034334887584939515, 0.03291509367959117, 0.03325243646690979, 0.03226665029085164, 0.03275214068403131, 0.0360022415822107, 0.03663308200482874, 0.034491706603143844, 0.0338068641820707, 0.03275361843520383, 0.03600305376965098, 0.03683943699143153, 0.03718343175764788, 0.03466253256400984, 0.035570337794891445, 0.036471463316681864, 0.03717420582883128, 0.03783667637903175, 0.038286596603684, 0.03485464210089554, 0.036262260926746316, 0.03711303718723162, 0.03572393353222253, 0.03583774633183837, 0.03456882478160373, 0.03329146690260813, 0.03444272099734688, 0.03452985467990668, 0.03375885119909169, 0.03508816258046259, 0.03445748710264468, 0.03294778143947462, 0.03183069446317984, 0.0358676721137433, 0.03263340016247513])

## Setup: cosmology, box, lines and skewers

In [ ]:
cosmology = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
L_box, z_box = 2.0 * 110.7, 2.2                        # [cMpc]
N_grid_skewers, N_ppsk, N_v = 32, 2048, 512             # skewers per side, points per skewer, velocity pixels
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
# other options: "SiII-1190", "SiII-1193"
colors = pt._neon_colors(line_bundle.species_keys)
ii_skewer = N_grid_skewers ** 2 // 2

def make_skewers(n_grid):
    pts = jnp.reshape(aux.make_skewer_grid_points(L_box, n_grid, N_ppsk, direction="y"), (n_grid ** 2, N_ppsk, 3))
    return SkewerBundle(spatial_points=pts, cosmologies=cosmology, a0=1 / (1 + z_box), x0=L_box)

skewer_bundle = make_skewers(N_grid_skewers)
a_dagger = 1 / (1 + z_box)
a_em = skewer_bundle.scale_factors[0, 0, 0]
v_max = phys.c_kms * jnp.log(a_dagger / a_em)
velocities_kms = jnp.linspace(0, v_max, N_v)
dv = float(velocities_kms[1] - velocities_kms[0])

## Fields: GRF densities
The density is n = 10^(log_mid + log_amp·tanh(δ/1.2)), mapped into [10⁻¹⁶, n0_max]. Here δ is a unit-variance `PowerLawGRF` on a 512³ grid with P(k) ∝ (k/k₀)^(n_s−1), n_s = −1.25.
- **Cutoff:** the previous version filtered the spectrum with exp(−(k/50)²). `PowerLawGRF` has no cutoff, but the factor is ≥ 0.98 below the grid Nyquist (7.3 cMpc⁻¹).
- **Shared realisation:** every species uses the same realisation (same key) and the same temperature. Si III and Si II have n0_max scaled by 1/250 and 1/150.
- **Species:** fields are built only for the species in `line_bundle`. The previous version also defined SiII, which `OpticalDepthCalculator` now rejects when the bundle has no SiII line.

In [ ]:
N_GRID_FIELDS = 512
N0_RATIO = {"HI": 1.0, "SiIII": 1 / 250, "SiII": 1 / 150}      # n0_max relative to HI

def build_fields(n_s=-1.25, T0_HI=15370.0, n0_max_HI=1.35e-11, key=jax.random.PRNGKey(0)):
    """Fields of every species in `line_bundle` (one shared GRF realisation) and their samplers."""
    grf = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=n_s, L=(L_box,) * 3, N=(N_GRID_FIELDS,) * 3)
    samplers = {sp: toy_samplers.number_density_sampler_gaussian_3d(key, grf, n0_min=1e-16,
                                                                    n0_max=n0_max_HI * N0_RATIO[sp], contrast=1.2)
                for sp in line_bundle.species_keys}
    fields = Fields({sp: {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_HI, T0_HI, L_box)),    # [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),              # [km/s]
        "number_density": ScalarField(samplers[sp]),                                          # [cm^-3]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),               # [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),             # [GHz]
    } for sp in line_bundle.species_keys})
    return fields, grf, samplers

fields, grf, samplers = build_fields()

quad = np.asarray(skewer_bundle.quadri_vectors[0, ii_skewer])
n_skewer = np.asarray(fields.number_density(quad))                     # (N_species, N_ppsk)
fig, ax = pt.plot_neon_lines(quad[:, 2], dict(zip(line_bundle.species_keys, n_skewer)), logy=True,
                             xlabel=r"LoS position $[\mathrm{cMpc}]$", ylabel=r"$n\ [\mathrm{cm^{-3}}]$")
plt.show()

### P1D of the density contrast along the skewers
δ_n = n/⟨n⟩ − 1 of HI along all skewers, in comoving units. The previous version passed log₁₀ n to `DeltaFieldBuilder`, which treats its input as an optical depth, so that figure showed exp(−log₁₀ n) instead. The spikes at k ≈ 14.5 cMpc⁻¹ and 29 cMpc⁻¹ are 2π/Δx and its harmonic, where Δx = L_box/512 is the GRF grid spacing. They come from trilinear interpolation of the 512³ grid onto 2048 points per skewer. The GRF itself is checked against its input power law in the last section.

In [ ]:
as_overflux = lambda d: SimpleNamespace(subset_keys=["a"], delta_terms={"a": d}, C_per_sample=jnp.zeros((d.shape[0], 1)), total=d)
n_HI = np.asarray(fields.number_density(np.asarray(skewer_bundle.quadri_vectors[0])))[0]   # (N_sk, N_ppsk)
st_n = OverfluxStatistics1D(as_overflux(jnp.asarray(n_HI / n_HI.mean(axis=1, keepdims=True) - 1.0)), L_box / N_ppsk).compute_p1d()
fig, ax = plt.subplots(figsize=(12, 6), layout="constrained")
ax.loglog(np.asarray(st_n.k)[1:], np.asarray(st_n.Pk_total)[1:], lw=3, color="k")
ax.set(xlabel=r"$k\ [\mathrm{cMpc^{-1}}]$", ylabel=r"$P_{\rm 1D}[\delta_n]\ [\mathrm{cMpc}]$")
plt.show()

## Optical depth
τ(v) of Lyα and Si III 1206 along every skewer, from the GRF fields, wrapped onto the Lyα frame. The previous version could replace this τ with Dirac-delta spikes (not a GRF field). That option is removed: the GRF τ already shows the cross-term signatures, and the linear-regime metals case in the last section shows them exactly.

In [ ]:
tau = od.OpticalDepthCalculator(skewer_bundle=skewer_bundle, velocities_kms=velocities_kms,
                                line_bundle=line_bundle, fields=fields, a_dagger=a_dagger)
tau.compute()
tau.wrap_to_ref_line()
optical_depth = tau.optical_depth
print("mean transmission per line:", {l.id: round(float(jnp.mean(jnp.exp(-optical_depth[i]))), 4)
                                     for i, l in enumerate(line_bundle.lines)})

## Overflux and the spectra of one skewer

In [ ]:
overflux = DeltaFieldBuilder(optical_depth).build()
F = np.exp(-np.asarray(optical_depth[:, ii_skewer]))
fig, ax = pt.plot_neon_lines(np.asarray(velocities_kms), {l.id: F[i] for i, l in enumerate(line_bundle.lines)},
                             total=F.prod(axis=0), colors={l.id: colors[l.species] for l in line_bundle.lines},
                             vlines=(0.0, float(v_max)), ylim=(-0.05, 1.05),
                             xlabel=r"wrapped velocity $v\ [\mathrm{km\,s^{-1}}]$", ylabel="transmission")
plt.show()

## 1D statistics by line pair and by order
These figures use the same views as notebook 8:
- pair terms per order, and the sum of the weighted pair terms over the total, for P and for ξ;
- the exact orders superimposed, for kP/π and for ξ1D.

The Lyα–Si III separation is marked on the ξ panel. Label "1" is Lyα and "2" is Si III.

In [ ]:
delta_v_list = [float(d) for d in np.asarray(line_bundle.delta_v)]
velocity_domain = float(velocities_kms[-1] - velocities_kms[0])
stats = OverfluxStatistics1D(overflux, dv).compute_p1d().compute_xi1d()
k = np.asarray(stats.k)
k_label = r"$k\ [\mathrm{s\,km^{-1}}]$"

fig, axes, axes_res, _ = pt.plot_p1d_by_order(
    stats.ordered_pairs, stats.Pk_pair, k, delta_v_list, velocity_domain, N_v, use_log_x=True,
    residual_mode="sum", show_harmonics=False, use_symlog_main=False, xlabel=k_label)
fig, _, _ = pt.plot_p1d_total_recovery(
    stats.Pk_total, stats.Pk_total_from_terms, stats.Pk_pair, k, delta_v_list, velocity_domain, N_v,
    show_harmonics=False, show_order_sums=True, show_order_sum_residuals=True, main_symlog_linthresh=1.0,
    use_symlog_residual=False, xlabel=k_label)
fig, axs = pt.plot_orders_superimposed(stats, "P", k_weighted=True, yscale="linear")
axs[0].set_xlim(right=0.1)
fig, _, _, _ = pt.plot_xi_total_recovery(
    stats.xi_total, stats.xi_total_from_terms, stats.xi_pair, np.asarray(stats.lags), delta_v_list, velocity_domain,
    use_symlog_main=False, use_symlog_residual=False, show_half_line=False, annotate_half_line=False,
    xlim=(0.0, 4000.0), xlabel=r"$\Delta v\ [\mathrm{km\,s^{-1}}]$")
fig, axs = pt.plot_orders_superimposed(stats, "xi", delta_v_list=delta_v_list, max_lag=4000.0)
plt.show()

## Comparison with DESI DR1

In [ ]:
model_k, model_y = k[1:], np.asarray(stats.Pk_total)[1:] * k[1:] / np.pi
residuals = 100.0 * (model_y / np.interp(model_k, Jonas2026_x, Jonas2026_y) - 1.0)
fig, (ax, ax_res) = plt.subplots(2, 1, figsize=(12, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1]}, layout="constrained")
ax.plot(model_k, model_y, lw=4.0, color=colors["SiIII"], label="SkewerJAX")
ax.plot(Jonas2026_x, Jonas2026_y, lw=3.0, color="red", label=r"Chaves-Montero et al. (2026), $z=2.2$")
ax.set(yscale="log", ylabel=r"$k P_{\rm 1D}/\pi$", xlim=(0.0, 0.027), ylim=(0.007, 0.06))
ax.legend(fontsize="small")
ax_res.plot(model_k, residuals, lw=2.0, color=colors["SiIII"])
ax_res.axhline(0, color="k", lw=1.5, ls="--")
ax_res.fill_between(model_k, -5, 5, color="grey", alpha=0.3)
ax_res.set(xlabel=k_label, ylabel=r"residuals [%]", ylim=(-20, 20))
plt.show()

## Parameter sweep
The sweep varies one parameter at a time around the fiducial (n_s, T₀^HI, n0_max^HI) and compares kP/π with DESI.
- **Grid:** to keep the runtime short, it uses a coarser `SWEEP_N_GRID`² skewer grid. Set it to `N_grid_skewers` for the full resolution, about 4× slower.
- **Seed:** the GRF key is fixed, so only the parameters change.

In [ ]:
SWEEP_N_GRID = 16
sweep_skewers = make_skewers(SWEEP_N_GRID)

def run_simulation(n_s, T0_HI, n0_max_HI):
    flds, _, _ = build_fields(n_s=n_s, T0_HI=T0_HI, n0_max_HI=n0_max_HI)
    t = od.OpticalDepthCalculator(skewer_bundle=sweep_skewers, velocities_kms=velocities_kms,
                                  line_bundle=line_bundle, fields=flds, a_dagger=a_dagger)
    t.compute()
    t.wrap_to_ref_line()
    s = OverfluxStatistics1D(DeltaFieldBuilder(t.optical_depth).build(), dv).compute_p1d()
    kk = np.asarray(s.k)[1:]
    return kk, np.asarray(s.Pk_total)[1:] * kk / np.pi

fiducial = {"n_s": -1.25, "T0_HI": 15370.0, "n0_max_HI": 1.35e-11}
variations = {"n_s": [-3.0, -2.0, -1.25, 0.0, 1.0, 2.0, 3.0],
              "T0_HI": [5000.0, 10000.0, 15370.0, 20000.0, 40000.0],
              "n0_max_HI": [1.0e-12, 5.0e-12, 1.35e-11, 5e-11, 1e-10]}
sweep = {p: {val: run_simulation(**{**fiducial, p: val}) for val in vals} for p, vals in variations.items()}

cmaps = {"n_s": "berlin", "T0_HI": "inferno", "n0_max_HI": "cividis"}
param_labels = {"n_s": r"$n_s$", "T0_HI": r"$T_{0,\rm HI}\ [\mathrm{K}]$", "n0_max_HI": r"$n^{\rm HI}_{0,\max}\ [\mathrm{cm^{-3}}]$"}
for p, vals in variations.items():
    fig, (ax, ax_res) = plt.subplots(2, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1]}, layout="constrained")
    norm, cmap = mcolors.Normalize(min(vals), max(vals)), plt.get_cmap(cmaps[p])
    for val in vals:
        mk, my = sweep[p][val]
        ax.plot(mk, my, lw=3.0, color=cmap(norm(val)))
        ax_res.plot(mk, 100.0 * (my / np.interp(mk, Jonas2026_x, Jonas2026_y) - 1.0), lw=2.0, color=cmap(norm(val)))
    ax.plot(Jonas2026_x, Jonas2026_y, lw=3.0, color="red", label="Chaves-Montero et al. (2026)")
    ax.set(yscale="log", ylabel=r"$k P_{\rm 1D}/\pi$", xlim=(0.0, 0.027))
    ax.legend(fontsize="small", loc="upper right")
    ax_res.axhline(0, color="k", lw=1.5, ls="--")
    ax_res.fill_between(mk, -5, 5, color="grey", alpha=0.3)
    ax_res.set(xlabel=k_label, ylabel=r"residuals [%]", ylim=(-60, 60))
    fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=[ax, ax_res], label=param_labels[p])
    plt.show()

## Metal contamination models
Both models are fitted to the ratio P_tot/P_Lyα for k < 0.1 s/km, where P_Lyα is the raw Lyα auto term P₁₁ and Δv is the line separation from `line_bundle` (`models/models.py`):
- **McDonald:** P_tot/P_Lyα = 1 + a² + 2a cos(kΔv).
- **Ma et al.:** amplitudes that depend on scale, a_auto e^{k/k_auto} + a_cross cos(kΔv) e^{k/k_cross}. It contains McDonald as a limit, so its χ² must not be larger.

In [ ]:
p_lya, p_tot = np.asarray(stats.Pk_pair[("1", "1")]), np.asarray(stats.Pk_total)
delta_v_siiii = abs(delta_v_list[1])
k_fit_range = (0.0, 0.1)
fit_mcd = models.fit_mcdonald_a(k=k, p_tot=p_tot, p_lya=p_lya, delta_v=delta_v_siiii, a_bounds=(-10.2, 10.2),
                                k_range=k_fit_range, fit_mode="ratio", optimizer_options={"xatol": 1e-14, "maxiter": 5000})
fit_ma = models.fit_ma_model_profiled(
    k=k, p_tot=p_tot, p_lya=p_lya, delta_v=delta_v_siiii, k_range=k_fit_range, fit_mode="ratio",
    lambda_bounds={"lambda_auto": (0.0, 500.0), "lambda_cross_abs": (0.0, 500.0)},    # 0 = McDonald limit
    amplitude_bounds={"a_auto": (0.0, 100.0), "a_cross": (-100.0, 100.0)}, enforce_positive_model=False,
    linear_lsq_tol=1e-14, run_global=True,
    global_optimizer_options={"tol": 1e-10, "atol": 1e-12, "maxiter": 1000, "popsize": 25, "polish": False,
                              "seed": 12345, "workers": 1, "updating": "immediate"},
    polish_optimizer_options={"maxiter": 50000, "ftol": 1e-15, "gtol": 1e-13, "maxls": 100})
ma_par = {p: fit_ma[p] for p in ("a_auto", "a_cross", "k_auto", "k_cross")}
ratio_mcd = models.mcdonald_factor(k=k, a=fit_mcd["a"], delta_v=delta_v_siiii)
ratio_ma = models.ma_factor(k=k, **ma_par, delta_v=delta_v_siiii)
print(f"McDonald: a = {fit_mcd['a']:.4e}, chi2 = {fit_mcd['chi2']:.4e} ({fit_mcd['dof']} dof)")
print("Ma et al.: " + ", ".join(f"{p} = {v:.3e}" for p, v in ma_par.items()) + f", chi2 = {fit_ma['chi2']:.4e}")

m = (k > k_fit_range[0]) & (k < k_fit_range[1]) & (p_lya > 0)
km, pl, ptt = k[m], p_lya[m], p_tot[m]
mcd_label = rf"McDonald: $a={fit_mcd['a']:.3e}$"
ma_label = (r"Ma et al.: " rf"$a_{{\rm auto}}={ma_par['a_auto']:.2e}$, $a_{{\rm cross}}={ma_par['a_cross']:.2e}$, "
            rf"$k_{{\rm auto}}={ma_par['k_auto']:.2e}$, $k_{{\rm cross}}={ma_par['k_cross']:.2e}$")
cmap = plt.get_cmap("tab10")
for y_data, y_mcd, y_ma, ylab in [(ptt * km / np.pi, ratio_mcd[m] * pl * km / np.pi, ratio_ma[m] * pl * km / np.pi, r"$k P_{\rm 1D}/\pi$"),
                                  (ptt / pl, ratio_mcd[m], ratio_ma[m], r"$P_{\rm tot}/P_{\rm Ly\alpha}$")]:
    fig, (ax, axr) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={"height_ratios": [3.5, 1.0]}, layout="constrained")
    ax.plot(km, y_data, lw=1.5, color="k", label="total")
    if ylab.startswith("$k"):
        ax.plot(km, pl * km / np.pi, lw=1.5, color=cmap(0), label=r"Ly$\alpha$ only")
    else:
        ax.axhline(1.0, lw=1.2, color="0.5")
    ax.plot(km, y_mcd, lw=2.5, ls="--", color=cmap(1), label=mcd_label)
    ax.plot(km, y_ma, lw=3.0, ls=":", color=cmap(2), label=ma_label)
    ax.set_ylabel(ylab)
    ax.legend(fontsize="x-small")
    axr.plot(km, y_mcd / y_data - 1, lw=2.5, ls="--", color=cmap(1))
    axr.plot(km, y_ma / y_data - 1, lw=3.0, ls=":", color=cmap(2))
    axr.axhline(0.0, lw=1.2, color="0.5")
    axr.set(xlabel=k_label, ylabel="model/data − 1", xlim=k_fit_range)
    plt.show()

## 3D statistics
The 32×32 skewers form a regular grid, with a transverse spacing of L_box/32 [cMpc] and dv [km/s] along the line of sight. The figures show:
- P3D in (k⊥, k∥) cells (`bin_p3d_kperp`), in mixed units;
- ξ3D binned in r⊥ (`bin_xi3d_rperp`) as a heatmap, profiles per r⊥ bin, and a 3D surface (symlog).

In [ ]:
stats3d = OverfluxStatistics3D(overflux, dv, L_box / N_grid_skewers, grid_shape=(N_grid_skewers,) * 2)
fig, ax = pt.plot_p3d_kperp_kpar(stats3d.bin_p3d_kperp(n_kperp_bins=6), units=pt.UNITS_3D)
plt.show()
fig, axs = pt.plot_xi3d_rperp_pi(stats3d.bin_xi3d_rperp(8), scale="symlog")
plt.show()

## Input vs recovered
The level of each comparison determines what has to match.

**1. GRF level: exact in expectation.**
- **What is compared:** the density fields come from realisation 0 of `grf`. Here `N_REAL` realisations of the same `grf` (realisation 0 included) are measured directly, as δ itself rather than the normalised δ/σ that the sampler uses. They are compared with `grf.power` and with its discrete inverse transform ξ_in = irfftn(P_in)/dV.
- **Estimators:** `OverfluxStatistics3D` computes P̂ = (dV/N)|rfftn δ|² and ξ̂ = irfftn(P̂)/dV with no window and no smoothing.
- **Binning:** binned values are mode-weighted means, compared with the input binned the same way (`bin_p3d_kmod(…, Pk3d_input=P_in)`).
- **Errors:** for P they are Gaussian, σ = P √(2/(n_modes N_REAL)); for ξ they come from the scatter over realisations.

**2. Physical flux level (the spectra and statistics above): no input match is expected, even for small fluctuations.** Between δ and δ_F the pipeline applies:
- the local density map n = 10^(log_mid + log_amp tanh(δ/σ/1.2)), which is non-linear;
- trilinear interpolation of the GRF grid onto the skewer points, a window that also aliases (the spikes at 2π/Δx in notebook 9);
- thermal Doppler broadening, a Gaussian line profile with b = √(2k_BT/m), which filters τ by e^{−k²b²/4} (`optical_depth/line_profile.py`; notes `sec_flux.tex`, Eq. line_profile);
- the a⁻² factor in dτ/dx and the light-cone mapping v(x) = c ln(a†/a(x)), which together add a slow trend along the skewer;
- `wrap_to_ref_line` for the non-reference lines, a linear interpolation by a non-integer pixel shift that suppresses power by 1 − 2f(1−f)(1 − cos k dv);
- F = e^{−τ} and the normalisation by ⟨F⟩, with τ ≳ 1.

**3. Flux level, linear regime with metals: exact to first order.** This isolates the flux mapping and the line separation.
- **Fields:** a 1D velocity-space GRF with σ_δ = 0.01. Its pixel is chosen so that the Lyα–Si III separation Δv from `LineBundle` is exactly 40 pixels, and n_s = 0.5 gives a sharp ξ peak. The optical depths are τ_Lyα = τ₀ e^{δ(v)} and τ_SiIII = a τ₀ e^{δ(v−Δv)}.
- **Prediction:** to first order (notes `sec_flux.tex`, Eqs. flux and delta_j), δ_tot → −τ₀[δ(v) + a δ(v−Δv)], so

$$P_{\rm Ly\alpha}\to\tau_0^2P_{\rm in},\qquad P_{\rm tot}\to\tau_0^2\,\bigl[1+a^2+2a\cos(k\Delta v)\bigr]P_{\rm in}.$$

  This is exactly the McDonald model `models.mcdonald_factor` (Ma et al. Eq. Pcross_old; McDonald et al. §"The Correlation Function and the SiIII Cross-Correlation").
- **Consequences:** ξ_tot is its inverse transform, with peaks of relative height a at ±Δv, and a McDonald fit must return the input a.
- **Physical-pipeline case:** the cross-term signatures sit at the same Δv, but their amplitudes are not those of the linear model, for the reasons in point 2. There the McDonald and Ma et al. fits are effective descriptions, and only the positions of the signatures are checked.

**4. Decomposition:** Σ orders = Σ weighted terms = total is an algebraic identity (shown in the figures above). The notebook runs in float32, so the tolerance is 1e-4.

In [ ]:
N_REAL = 8
as_overflux = lambda d: SimpleNamespace(subset_keys=["a"], delta_terms={"a": d}, C_per_sample=jnp.zeros((d.shape[0], 1)), total=d)
keys3 = [jax.random.PRNGKey(0)] + list(jax.random.split(jax.random.PRNGKey(1), N_REAL - 1))   # 0 = the density field
(Nx, Ny, Nz), (dx3, dy3, dz3) = grf.shape, grf.dx
_, kmod3 = grf.k_grid(real=True)
P3_in = np.asarray(grf.power(kmod3))
xi3_in = np.asarray(xi3d_from_power(P3_in, grf.shape, grf.cell_volume))
ked3 = np.geomspace(0.999 * min(grf.k_min), min(grf.k_max), 13)
P3_list, xi3_list = [], []
for kk in keys3:
    st3 = OverfluxStatistics3D(as_overflux(grf.sample(kk).reshape(Nx * Ny, Nz)), dz3, (dx3, dy3), grid_shape=(Nx, Ny)).compute_xi3d()
    P3_list.append(np.asarray(st3.bin_p3d_kmod(ked3)[0]))
    xi3_list.append(np.asarray(st3.xi3d_total))
_, kb3, _, nb3 = map(np.asarray, st3.bin_p3d_kmod(ked3))
P3_in_b = np.asarray(st3.bin_p3d_kmod(ked3, Pk3d_input=P3_in)[0])
P3_mean, _ = pt.sample_mean_and_error(P3_list)
xi3_mean, xi3_err = pt.sample_mean_and_error(xi3_list)
del xi3_list
fig, axs = pt.plot_p3d_recovery((P3_mean, kb3, ked3, nb3), P3_in_b, power=grf.power, n_real=N_REAL,
                                label=f"GRF, {N_REAL} realisations", units=pt.UNITS_3D_COMOVING)
plt.show()
fig, axs = pt.plot_xi3d_recovery(st3, xi3_in, xi=xi3_mean, sigma=xi3_err, max_lag=40.0, units=pt.UNITS_3D_COMOVING)
plt.show()


In [ ]:
# flux level, linear regime with metals: 1D velocity-space GRF, two lines separated by Delta v from LineBundle
dv_sep = float(np.asarray(line_bundle.delta_v)[1])                     # signed Si III - Lya separation [km/s]
SHIFT, N_V_LIN, N_SK_LIN, TAU0, A_SIIII = 40, 512, 2048, 1.0, 0.1
dv_lin = abs(dv_sep) / SHIFT                                            # |Delta v| = 40 pixels exactly
g_unit = PowerLawGRF(1.0, 0.01, 0.5, L=N_V_LIN * dv_lin, N=N_V_LIN)
grf_v = PowerLawGRF(1e-4 / float(g_unit.variance()), 0.01, 0.5, L=N_V_LIN * dv_lin, N=N_V_LIN)   # sigma_delta = 0.01
d_v = jax.vmap(grf_v.sample)(jax.random.split(jax.random.PRNGKey(7), N_SK_LIN))
tau_v = jnp.stack([TAU0 * jnp.exp(d_v), A_SIIII * TAU0 * jnp.exp(jnp.roll(d_v, int(np.sign(dv_sep)) * SHIFT, axis=-1))])
st_v = OverfluxStatistics1D(DeltaFieldBuilder(tau_v).build(), dv_lin).compute_p1d().compute_xi1d()
k_v = np.asarray(st_v.k)
P_lya_exp = TAU0 ** 2 * np.asarray(grf_v.power(k_v))
P_tot_exp = models.mcdonald_factor(k_v, A_SIIII, delta_v=abs(dv_sep)) * P_lya_exp
edges_v = pt.hybrid_k_edges(grf_v.k_min[0], grf_v.k_max[0], n_lin=N_V_LIN // 2, n_log=1)   # one k mode per bin
bins_v11, bins_vtot = st_v.bin_p1d(edges_v, Pk=st_v.Pk_pair[("1", "1")]), st_v.bin_p1d(edges_v)
exp_v11, exp_vtot = st_v.bin_p1d(edges_v, Pk=P_lya_exp)[0], st_v.bin_p1d(edges_v, Pk=P_tot_exp)[0]
fig, axs = pt.plot_pk_recovery(bins_v11, exp_v11, label=r"Ly$\alpha$ term $P_{11}$")
pt.plot_pk_recovery(bins_vtot, exp_vtot, label=r"total $P_{\rm tot}$", ax=axs)
axs[0].plot(k_v[1:], P_lya_exp[1:], color="k", lw=1.5, label=r"$\tau_0^2P_{\rm in}$")
axs[0].plot(k_v[1:], P_tot_exp[1:], color="k", lw=1.5, ls="--", label=r"$\tau_0^2[1+a^2+2a\cos k\Delta v]P_{\rm in}$ (McDonald)")
axs[0].legend(fontsize="x-small", frameon=False)
axs[0].set_xlim(right=0.03)
plt.show()
xi_v_exp = np.fft.fftshift(np.asarray(xi_from_power(P_tot_exp, N_V_LIN, dv_lin)))
_, xi_v_err = pt.sample_mean_and_error(st_v.xi_total_per_sample)
fig, axs = pt.plot_xi_recovery(st_v, xi_v_exp, sigma=xi_v_err, max_lag=4000.0, label=r"$\xi_{\rm tot}$, linear regime")
for a_ in axs:
    a_.axvline(dv_sep, color="C3", ls="--", lw=1); a_.axvline(-dv_sep, color="C3", ls="--", lw=1)
plt.show()
fit_v = models.fit_mcdonald_a(k=k_v, p_tot=np.asarray(st_v.Pk_total), p_lya=np.asarray(st_v.Pk_pair[("1", "1")]),
                              delta_v=abs(dv_sep), a_bounds=(-1.0, 1.0), k_range=(0.0, float(k_v[-1]) * 1.01),
                              fit_mode="ratio", optimizer_options={"xatol": 1e-12, "maxiter": 5000})
print(f"McDonald fit on the linear-regime case: a = {fit_v['a']:.5f} (input a = {A_SIIII})")

## Checks

In [ ]:
CHECKS = []
def check(name, passed, detail=""):
    CHECKS.append((name, bool(passed)))
    print(f"[{'PASS' if passed else 'FAIL'}] {name}: {detail}")
rel = lambda a, b: float(np.max(np.abs(np.asarray(a) - np.asarray(b))) / np.max(np.abs(np.asarray(b))))

same = rel(np.asarray(samplers["HI"].delta_grid) * np.sqrt(float(grf.variance())), grf.sample(keys3[0]))
check("GRF level: the density field is realisation 0 of grf (delta_grid * sigma)", same < 1e-5, f"rel {same:.1e}")
sig3 = pt.pk_sigma(P3_in_b, nb3, N_REAL)
pull3 = (P3_mean - P3_in_b) / sig3
check("GRF level: P3D(|k|) = input (chi2/dof < 2)", np.nanmean(pull3 ** 2) < 2 and np.nanmax(np.abs(pull3)) < 4,
      pt.recovery_summary(P3_mean, P3_in_b, sig3, "P3D"))
c0 = tuple(n // 2 for n in xi3_mean.shape)
dxi = float(np.max(np.abs(xi3_mean - xi3_in)) / xi3_in[c0])
frac = float(np.mean(np.abs(xi3_mean - xi3_in) < 2 * xi3_err))
check("GRF level: xi3D = inverse transform of the input (max |dxi| < 5% xi(0), >= 75% of cells within 2 sigma)",
      dxi < 0.05 and frac >= 0.75, f"max |dxi|/xi(0) = {dxi:.3f}, {100 * frac:.0f}% within 2 sigma ({N_REAL} realisations)")

def chi2_check(name, binned, P_exp):
    sig = pt.pk_sigma(P_exp, binned[3]); pull = (np.asarray(binned[0]) - np.asarray(P_exp)) / sig
    check(name, np.nanmean(pull ** 2) < 1.5 and np.nanmax(np.abs(pull)) < 5, pt.recovery_summary(binned[0], P_exp, sig, "P"))
chi2_check("flux level, linear: P_Lya -> tau0^2 P_in", bins_v11, exp_v11)
chi2_check("flux level, linear: P_tot -> tau0^2 [1 + a^2 + 2a cos(k dv)] P_in (McDonald, LineBundle dv)", bins_vtot, exp_vtot)
pull = (np.asarray(st_v.xi_total) - xi_v_exp) / xi_v_err
check("flux level, linear: xi_tot = inverse transform of the McDonald prediction",
      np.mean(np.abs(pull) < 2) >= 0.8 and np.max(np.abs(pull)) < 6, pt.recovery_summary(st_v.xi_total, xi_v_exp, xi_v_err, "xi", ratio=False))
lags_v = np.asarray(st_v.lags); far = np.abs(lags_v) > abs(dv_sep) / 2
peak_v = abs(lags_v[far][np.argmax(np.asarray(st_v.xi_total)[far])])
check("flux level, linear: xi_tot peaks at |dv| from LineBundle", abs(peak_v - abs(dv_sep)) <= dv_lin,
      f"peak at {peak_v:.1f} km/s, |dv| = {abs(dv_sep):.1f} km/s")
check("flux level, linear: McDonald fit recovers the input a (1%)", abs(fit_v["a"] / A_SIIII - 1) < 0.01,
      f"a_fit = {fit_v['a']:.5f}, a_in = {A_SIIII}")

P_terms = sum(np.asarray(stats.Pk_total_from_terms_contributions[p]) for p in stats.term_pairs)
check("decomposition, P1D: sum of orders = sum of terms = total",
      max(rel(sum(stats.Pk_total_by_order.values()), stats.Pk_total), rel(P_terms, stats.Pk_total)) < 1e-4,
      f"rel {rel(sum(stats.Pk_total_by_order.values()), stats.Pk_total):.1e}, {rel(P_terms, stats.Pk_total):.1e}")
check("decomposition, xi1D: sum of orders = total", rel(sum(stats.xi_total_by_order.values()), stats.xi_total) < 1e-4,
      f"rel {rel(sum(stats.xi_total_by_order.values()), stats.xi_total):.1e}")

P12, P22 = np.asarray(stats.Pk_pair[("1", "2")]), np.asarray(stats.Pk_pair[("2", "2")])
r = float(np.corrcoef(P12[m] / np.sqrt(pl * P22[m]), np.cos(km * delta_v_siiii))[0, 1])
check("physical pipeline: Re P_12 / sqrt(P_11 P_22) ~ cos(k dv)", r > 0.8, f"correlation coefficient {r:.3f}")
lags = np.asarray(stats.lags)
xi_x = np.fft.fftshift(np.asarray(stats.xi_pair[("1", "2")]) + np.asarray(stats.xi_pair[("2", "1")]))
far = np.abs(lags) > delta_v_siiii / 2
lag_peak = abs(lags[far][np.argmax(xi_x[far])])
check("physical pipeline: xi_12 + xi_21 peaks at |dv|", abs(lag_peak - delta_v_siiii) <= 2 * dv,
      f"peak at {lag_peak:.1f} km/s, |dv| = {delta_v_siiii:.1f} km/s, pixel {dv:.1f} km/s")
rms = lambda y: float(np.sqrt(np.mean((y - ptt / pl) ** 2)))
check("physical pipeline: McDonald reduces the ratio residual vs a = 0", rms(ratio_mcd[m]) < rms(1.0),
      f"rms {rms(ratio_mcd[m]):.3e} (a = {fit_mcd['a']:.3e}) vs {rms(1.0):.3e} (a = 0)")
check("physical pipeline: Ma et al. chi2 <= McDonald chi2 (nested)", fit_ma["chi2"] <= fit_mcd["chi2"] * (1 + 1e-9),
      f"{fit_ma['chi2']:.4e} vs {fit_mcd['chi2']:.4e}")
print(f"\nSUMMARY: {sum(p for _, p in CHECKS)}/{len(CHECKS)} checks passed")